# Práctica 2: Representación Vectorial de Texto mediante TF-IDF

## Objetivos
1. Reutilizar el pipeline de limpieza y preprocesamiento de la Práctica 1.
2. Recuperar la Bolsa de Palabras (Bag of Words) generada sobre el corpus de reseñas.
3. **Inciso a:** Implementar desde cero las funciones para el cálculo de TF (Term Frequency), IDF (Inverse Document Frequency) y TF-IDF sin utilizar librerías.
4. **Inciso b:** Programar la función para imprimir y visualizar la Matriz con los pesos TF-IDF resultantes.

In [1]:
import csv
import math
import string


class Preprocesa:
  """Clase para ejecutar el pipeline de normalización, limpieza y lematización

  diseñada en la Práctica 1.
  """

  def __init__(self):
    # Conjunto de caracteres de puntuación especiales a remover
    self.forbidden = {
        "?",
        "¿",
        "¡",
        "!",
        "'",
        '"',
        "‘",
        "’",
        "“",
        "”",
        "<",
        ">",
        "(",
        ")",
        ".",
        ",",
        ":",
        ";",
        "-",
        "&",
        "@",
        "/",
        "#",
        "$",
        "´",
        "`",
        "«",
        "»",
        "—",
        "–",
        "…",
        "•",
    }
    # Unión de signos estándar de string.punctuation con los caracteres personalizados
    self.all_punctuation = set(string.punctuation).union(self.forbidden)

    # Lista de palabras vacías (stopwords) en inglés a descartar
    self.stop_words = {
        "i",
        "me",
        "my",
        "myself",
        "we",
        "our",
        "ours",
        "ourselves",
        "you",
        "your",
        "yours",
        "yourself",
        "yourselves",
        "he",
        "him",
        "his",
        "himself",
        "she",
        "her",
        "hers",
        "herself",
        "it",
        "its",
        "itself",
        "they",
        "them",
        "their",
        "theirs",
        "themselves",
        "what",
        "which",
        "who",
        "whom",
        "this",
        "that",
        "these",
        "those",
        "am",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "have",
        "has",
        "had",
        "having",
        "do",
        "does",
        "did",
        "doing",
        "a",
        "an",
        "the",
        "and",
        "but",
        "if",
        "or",
        "because",
        "as",
        "until",
        "while",
        "of",
        "at",
        "by",
        "for",
        "with",
        "about",
        "against",
        "between",
        "into",
        "through",
        "during",
        "before",
        "after",
        "above",
        "below",
        "to",
        "from",
        "up",
        "down",
        "in",
        "out",
        "on",
        "off",
        "over",
        "under",
        "again",
        "further",
        "then",
        "once",
        "here",
        "there",
        "when",
        "where",
        "why",
        "how",
        "all",
        "any",
        "both",
        "each",
        "few",
        "more",
        "most",
        "other",
        "some",
        "such",
        "no",
        "nor",
        "not",
        "only",
        "own",
        "same",
        "so",
        "than",
        "too",
        "very",
        "s",
        "t",
        "can",
        "will",
        "just",
        "don",
        "should",
        "now",
    }

    # Diccionario léxico de lemas irregulares recurrentes en IMDB
    self.lemma_dict = {
        "ran": "run",
        "running": "run",
        "runs": "run",
        "saw": "see",
        "seen": "see",
        "seeing": "see",
        "sees": "see",
        "went": "go",
        "gone": "go",
        "going": "go",
        "goes": "go",
        "was": "be",
        "were": "be",
        "been": "be",
        "being": "be",
        "am": "be",
        "is": "be",
        "are": "be",
        "had": "have",
        "has": "have",
        "having": "have",
        "did": "do",
        "done": "do",
        "does": "do",
        "better": "good",
        "best": "good",
        "worse": "bad",
        "worst": "bad",
        "movies": "movie",
        "films": "film",
        "characters": "character",
        "actors": "actor",
        "scenes": "scene",
        "acting": "act",
        "watched": "watch",
        "watching": "watch",
    }

  def lower_words(self, text):
    """Convierte el texto completo a minúsculas."""
    return str(text).lower()

  def remove_tags(self, text):
    """Elimina etiquetas <br /> del dataset."""
    while "<" in text and ">" in text:
      start = text.find("<")
      end = text.find(">", start)
      if end == -1:
        break
      text = text[:start] + " " + text[end + 1 :]
    return text

  def remove_punctuation(self, text):
    """Filtra y elimina signos de puntuación, consolidando espacios múltiples."""
    punctuationfree = "".join(
        [char for char in text if char not in self.all_punctuation]
    )
    return " ".join(punctuationfree.split()).strip()

  def tokenize(self, text):
    """Segmenta la cadena en tokens alfanuméricos contiguos."""
    tokens, current = [], []
    for char in text:
      if char.isalnum():
        current.append(char)
      else:
        if current:
          tokens.append("".join(current))
          current = []
    if current:
      tokens.append("".join(current))
    return tokens

  def remove_stopwords(self, tokens):
    """Descarta los tokens que coinciden con la lista de stopwords."""
    return [p for p in tokens if p.lower() not in self.stop_words]

  def lemmatize_word(self, word):
    """Aplica reglas morfológicas y consultas de diccionario para reducir a la forma base."""
    if word in self.lemma_dict:
      return self.lemma_dict[word]
    if word.endswith("ies") and len(word) > 4:
      return word[:-3] + "y"
    elif word.endswith("es") and len(word) > 3:
      return word[:-2]
    elif word.endswith("s") and not word.endswith("ss") and len(word) > 3:
      return word[:-1]
    elif word.endswith("ed") and len(word) > 4:
      return word[:-2]
    elif word.endswith("ing") and len(word) > 5:
      return word[:-3]
    return word

  def apply_lemmatization(self, tokens):
    """Aplica la lematización a cada término de la lista de tokens."""
    return [self.lemmatize_word(w) for w in tokens]

  def pipeline(self, text):
    """Ejecuta en secuencia el flujo completo de preprocesamiento."""
    t_clean = self.remove_tags(self.lower_words(text))
    t_nopunct = self.remove_punctuation(t_clean)
    tokens = self.tokenize(t_nopunct)
    tokens_clean = self.remove_stopwords(tokens)
    res = self.apply_lemmatization(tokens_clean)
    return " ".join(res)


# Instanciación del módulo de preprocesamiento
prep = Preprocesa()
print("Clase de preprocesamiento inicializada correctamente.")

Clase de preprocesamiento inicializada correctamente.


## Carga del Dataset y Recuperación de la Bolsa de Palabras (Práctica 1)
Cargamos el archivo CSV de IMDB y ejecutamos el pipeline sobre la muestra para obtener las reseñas limpias (`clean_reviews`) y la Bolsa de Palabras (`vocab_counter`) mediante `collections.Counter`.

In [2]:
from collections import Counter
import csv
import os

# 1. Definición del nombre del archivo y búsqueda automática de la ruta
nombre_archivo = "IMDB Dataset.csv"

# Si no está en la raíz, revisa si está en carpetas comunes (p. ej. 'data/', '../', etc.)
rutas_posibles = [
    nombre_archivo,
    os.path.join("data", nombre_archivo),
    os.path.join("..", nombre_archivo),
    os.path.join("/content", nombre_archivo),  # Ruta estándar de Colab
]

archivo_kaggle = None
for ruta in rutas_posibles:
  if os.path.exists(ruta):
    archivo_kaggle = ruta
    break

# Si no se encuentra automáticamente, muestra los archivos disponibles en el directorio
if archivo_kaggle is None:
  archivos_en_directorio = os.listdir(".")
  raise FileNotFoundError(
      f"No se encontró '{nombre_archivo}'. Asegúrate de moverlo a esta carpeta.\n"
      f"Archivos encontrados en la carpeta actual: {archivos_en_directorio}"
  )

print(f"Archivo localizado con éxito en: {archivo_kaggle}")

# 2. Lectura del archivo CSV
reviews = []
with open(archivo_kaggle, mode="r", encoding="utf-8") as f:
  reader = csv.DictReader(f)
  for row in reader:
    reviews.append(row["review"])

# 3. Tamaño de muestra utilizado en la Práctica 1
n_muestras = 2500
clean_reviews = []

# Objeto Counter que almacenará la Bolsa de Palabras (Bag of Words)
vocab_counter = Counter()

print(f"Preprocesando {n_muestras} reseñas de IMDB...")

# Procesamiento iterativo de cada documento usando el pipeline
for r in reviews[:n_muestras]:
  texto_proc = prep.pipeline(r)
  clean_reviews.append(texto_proc)
  # Actualizamos las frecuencias globales de la bolsa de palabras
  vocab_counter.update(texto_proc.split())

print("Pipeline finalizado con éxito.")
print(f"Total de términos únicos en la Bolsa de Palabras: {len(vocab_counter)}")
print(
    "\nTop 10 términos más frecuentes de la bolsa de palabras:",
    vocab_counter.most_common(10),
)

Archivo localizado con éxito en: IMDB Dataset.csv
Preprocesando 2500 reseñas de IMDB...
Pipeline finalizado con éxito.
Total de términos únicos en la Bolsa de Palabras: 26719

Top 10 términos más frecuentes de la bolsa de palabras: [('movie', 5063), ('film', 4643), ('good', 2599), ('one', 2474), ('see', 2370), ('like', 1869), ('watch', 1414), ('character', 1380), ('go', 1376), ('get', 1299)]


## Inciso a) Funciones para Calcular TF-IDF desde Cero
Implementación matemática de los componentes sin librerías externas:
1. **TF (Term Frequency):** Proporción del término $t$ dentro del documento $d$:
   $$\text{TF}(t, d) = \frac{f_{t, d}}{\text{total de términos en } d}$$
2. **IDF (Inverse Document Frequency):** Ponderación de rareza en el corpus con suavizado:
   $$\text{IDF}(t, D) = \log_e\left(\frac{N}{\text{DF}(t) + 1}\right) + 1$$
3. **TF-IDF:** Multiplicación de ambos componentes para cada término de la Bolsa de Palabras.

In [3]:
def obtener_vocabulario_bolsa(vocab_counter, top_n=None):
  """Extrae los términos de la Bolsa de Palabras generada en la Práctica 1.

  Permite limitar a los 'top_n' más frecuentes para evitar matrices dispersas
  gigantescas.
  """
  if top_n is not None:
    # Retorna las palabras ordenadas por mayor frecuencia en la bolsa
    return [palabra for palabra, _ in vocab_counter.most_common(top_n)]
  # Retorna el vocabulario completo ordenado alfabéticamente
  return sorted(list(vocab_counter.keys()))


def calcular_tf(tokens_doc):
  """Calcula la frecuencia relativa de término (TF) para un documento tokenizado.

  TF(t, d) = (Frecuencia de t en d) / (Total de palabras en d)
  """
  total = len(tokens_doc)
  if total == 0:
    return {}

  # Conteo de ocurrencias de cada palabra en el documento
  conteo = {}
  for token in tokens_doc:
    conteo[token] = conteo.get(token, 0) + 1

  # Normalización por el total de palabras del documento
  return {termino: cant / total for termino, cant in conteo.items()}


def calcular_idf(corpus_tokens_set, vocabulario):
  """Calcula la frecuencia inversa de documento (IDF) para cada palabra de la bolsa.

  IDF(t) = ln(N / (DF(t) + 1)) + 1
  """
  N = len(corpus_tokens_set)
  idf_dict = {}

  for termino in vocabulario:
    # Contamos en cuántos documentos aparece el término (Document Frequency)
    df = sum(1 for doc_set in corpus_tokens_set if termino in doc_set)
    # Suavizado para evitar división entre cero y logaritmo de cero
    idf_dict[termino] = math.log(N / (df + 1)) + 1

  return idf_dict


def calcular_tfidf(corpus_limpio, vocab_counter, top_n=15):
  """a) Construye la matriz con los pesos TF-IDF integrando el corpus limpio

  y la Bolsa de Palabras (vocab_counter).
  """
  # 1. Obtenemos el vocabulario de referencia desde la bolsa de palabras
  vocabulario = obtener_vocabulario_bolsa(vocab_counter, top_n=top_n)

  # 2. Tokenizamos el corpus limpio tanto en listas como en sets (búsquedas O(1))
  corpus_tokens_list = [doc.split() for doc in corpus_limpio]
  corpus_tokens_set = [set(tokens) for tokens in corpus_tokens_list]

  # 3. Calculamos el vector global de valores IDF sobre el corpus
  idf_dict = calcular_idf(corpus_tokens_set, vocabulario)

  # 4. Cálculo iterativo de los pesos TF-IDF para cada documento
  matriz_tfidf = []
  for doc_tokens in corpus_tokens_list:
    tf_doc = calcular_tf(doc_tokens)
    fila_pesos = []
    for termino in vocabulario:
      # Si el término no está en el documento, TF es 0.0
      tf_val = tf_doc.get(termino, 0.0)
      idf_val = idf_dict[termino]
      # Multiplicación TF * IDF redondeada a 5 decimales
      peso_tfidf = tf_val * idf_val
      fila_pesos.append(round(peso_tfidf, 5))
    matriz_tfidf.append(fila_pesos)

  return matriz_tfidf, vocabulario


print("Funciones para el cálculo de TF-IDF compiladas correctamente.")

Funciones para el cálculo de TF-IDF compiladas correctamente.


## Inciso b) Función para Imprimir la Matriz con los Pesos TF/IDF
Se programa una función que da formato a la salida numérica en una tabla alineada, facilitando la inspección de los pesos asociados a las palabras más relevantes de la Bolsa de Palabras en los distintos documentos.

In [4]:
def imprimir_matriz_tfidf(matriz_tfidf, vocabulario, max_docs=10):
  """b) Imprime la matriz de pesos TF/IDF en formato tabular limpio."""
  col_width = 12

  # Creación de la fila de encabezado con los nombres de las palabras
  header = f"{'Reseña':<12}" + "".join(
      f"{termino:>{col_width}}" for termino in vocabulario
  )
  separador = "=" * len(header)

  # Impresión de títulos y líneas divisorias
  print(separador)
  print(header)
  print(separador)

  # Impresión fila por fila para los documentos seleccionados
  for i in range(min(max_docs, len(matriz_tfidf))):
    fila_valores = "".join(f"{peso:>{col_width}.5f}" for peso in matriz_tfidf[i])
    print(f"Doc {i+1:<8}{fila_valores}")

  print(separador)


# --- Ejecución de las funciones sobre los datos ---

# Seleccionamos los 12 términos más frecuentes de nuestra Bolsa de Palabras
top_k_terminos = 12

# a) Cálculo de TF-IDF utilizando la bolsa de palabras existente
matriz_tfidf, vocabulario_eval = calcular_tfidf(
    corpus_limpio=clean_reviews,
    vocab_counter=vocab_counter,
    top_n=top_k_terminos,
)

# b) Despliegue e impresión de la matriz para las primeras 10 reseñas
print(
    f"\nMatriz con los Pesos TF/IDF (Top {top_k_terminos} términos de la Bolsa"
    " de Palabras):"
)
imprimir_matriz_tfidf(matriz_tfidf, vocabulario_eval, max_docs=10)


Matriz con los Pesos TF/IDF (Top 12 términos de la Bolsa de Palabras):
Reseña             movie        film        good         one         see        like       watch   character          go         get         bad        even
Doc 1            0.00000     0.00000     0.00000     0.00967     0.00984     0.00000     0.03579     0.00000     0.02432     0.02471     0.00000     0.00000
Doc 2            0.00000     0.01793     0.00000     0.01923     0.01957     0.00000     0.02372     0.00000     0.00000     0.00000     0.00000     0.00000
Doc 3            0.00000     0.00000     0.00000     0.01878     0.01912     0.00000     0.02316     0.02485     0.02361     0.00000     0.00000     0.02412
Doc 4            0.06223     0.04366     0.00000     0.00000     0.02383     0.02623     0.02887     0.00000     0.02943     0.00000     0.00000     0.00000
Doc 5            0.01145     0.02410     0.03866     0.07754     0.01315     0.00000     0.01594     0.03419     0.00000     0.01651     0.0000

## Interpretación de Resultados
- **Valores iguales a 0.00000:** Indican que la palabra correspondiente de la Bolsa de Palabras no apareció dentro de ese documento específico ($\text{TF} = 0$).
- **Valores positivos:** Representan la ponderación de importancia de la palabra dentro de esa reseña; a mayor frecuencia local y mayor selectividad en el corpus global, mayor es el valor del peso resultante.
- **Uso directo de la Bolsa de Palabras:** Al enlazar `vocab_counter` con el cálculo de TF-IDF, se evalúa únicamente el espacio vectorial que fue caracterizado previamente en la Práctica 1.